In [66]:
import planetary_computer
import pystac_client
from IPython.display import display, Markdown

catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)
collection = catalog.get_collection("sentinel-3-slstr-lst-l2-netcdf")

display(Markdown(f"### {collection.id}\n\n{collection.description}"))

### sentinel-3-slstr-lst-l2-netcdf

This Collection provides Sentinel-3 [SLSTR Level-2 Land Surface Temperature](https://sentinel.esa.int/web/sentinel/user-guides/sentinel-3-slstr/product-types/level-2-lst) products containing data on land surface temperature measurements on a 1km grid. Radiance is measured in two channels to determine the temperature of the Earth's surface skin in the instrument field of view, where the term "skin" refers to the top surface of bare soil or the effective emitting temperature of vegetation canopies as viewed from above.

## Data files

The dataset includes data on the primary measurement variable, land surface temperature, in a single NetCDF file, `LST_in.nc`. A second file, `LST_ancillary.nc`, contains several ancillary variables:

- Normalized Difference Vegetation Index
- Surface biome classification
- Fractional vegetation cover
- Total water vapor column

In addition to the primary and ancillary data files, a standard set of annotation data files provide meteorological information, geolocation and time coordinates, geometry information, and quality flags. More information about the product and data processing can be found in the [User Guide](https://sentinels.copernicus.eu/web/sentinel/user-guides/sentinel-3-slstr/product-types/level-2-lst) and [Technical Guide](https://sentinel.esa.int/web/sentinel/technical-guides/sentinel-3-slstr/level-2/lst-processing).

This Collection contains Level-2 data in NetCDF files from April 2016 to present.

## STAC Item geometries

The Collection contains small "chips" and long "stripes" of data collected along the satellite direction of travel. Approximately five percent of the STAC Items describing long stripes of data contain geometries that encompass a larger area than an exact concave hull of the data extents. This may require additional filtering when searching the Collection for Items that spatially intersect an area of interest.


In [ ]:
bbox = [-74.01, 40.75, -73.86, 40.88]
time_window = "2021-06-01/2021-09-01"

In [68]:
import xarray as xr
import fsspec

search = catalog.search(
    bbox=bbox, 
    datetime=time_window,
    collections=["sentinel-3-slstr-lst-l2-netcdf"],
    query={"eo:cloud_cover": {"lt": 30}},
)

items = list(search.get_items())
print('This is the number of scenes that touch our region:',len(items))


C:\Users\16539\AppData\Roaming\Python\Python310\site-packages\pystac_client\item_search.py:826: FutureWarning: get_items() is deprecated, use items() instead
  warnings.warn(


This is the number of scenes that touch our region: 21


In [69]:
import rich.table

item = next(search.items())

t = rich.table.Table("Key", "Value")
for key, asset in item.assets.items():
    t.add_row(key, asset.description)

t

┏━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Key                ┃ Value                                                                         ┃
┡━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ lst-in             │ Land Surface Temperature (LST) values                                         │
│ slstr-met-tx       │ Meteorological parameters regridded onto the 16km tie points                  │
│ safe-manifest      │ SAFE product manifest                                                         │
│ slstr-time-in      │ Time annotations for the 1km grid                                             │
│ slstr-flags-in     │ Global flags for the 1km TIR grid, nadir view                                 │
│ lst-ancillary-ds   │ LST ancillary measurement dataset                                             │
│ slstr-indices-in   │ Scan, pixel and detector indices annotations for the 1km TIR grid, nadir view │
│ slstr-geodetic-in  │ Full resolution geodetic coordinates for the 1km TIR grid, nadir view         │
│ slstr-geodetic-tx  │ 16km geodetic coordinates                                                     │
│ slstr-geometry-tn  │ 16km solar and satellite geometry annotations, nadir view                     │
│ slstr-cartesian-in │ Full resolution cartesian coordinates for the 1km TIR grid, nadir view        │
│ slstr-cartesian-tx │ 16km cartesian coordinates                                                    │
└────────────────────┴───────────────────────────────────────────────────────────────────────────────┘

In [70]:
dataset = xr.open_dataset(fsspec.open(item.assets["lst-in"].href).open())
dataset

<xarray.Dataset>
Dimensions:                 (rows: 1200, columns: 1500, orphan_pixels: 187)
Dimensions without coordinates: rows, columns, orphan_pixels
Data variables:
    LST                     (rows, columns) float32 ...
    LST_orphan              (rows, orphan_pixels) float32 ...
    LST_uncertainty         (rows, columns) float32 ...
    LST_uncertainty_orphan  (rows, orphan_pixels) float32 ...
    exception               (rows, columns) int16 ...
    exception_orphan        (rows, orphan_pixels) int16 ...
Attributes: (12/16)
    absolute_orbit_number:  17380
    comment:                 
    contact:                eosupport@copernicus.esa.int
    creation_time:          20210828T012330Z
    history:                  2021-08-28T01:23:30Z: PUGCoreProcessor /data/ip...
    institution:            LN2
    ...                     ...
    source:                 IPF-SL-2 06.18
    start_offset:           54882
    start_time:             2021-08-26T15:40:26.116731Z
    stop_time:              2021-08-26T15:43:25.810293Z
    title:                  SLSTR Level 2 Product, Land Surface Temperature m...
    track_offset:           998

In [71]:
geo = xr.open_dataset(fsspec.open(item.assets["slstr-geodetic-in"].href).open()).load()
geo

<xarray.Dataset>
Dimensions:              (rows: 1200, columns: 1500, orphan_pixels: 187)
Dimensions without coordinates: rows, columns, orphan_pixels
Data variables:
    elevation_in         (rows, columns) float32 nan nan nan nan ... nan nan nan
    elevation_orphan_in  (rows, orphan_pixels) float32 204.0 204.0 ... nan nan
    latitude_in          (rows, columns) float64 52.48 52.47 ... 39.12 39.12
    latitude_orphan_in   (rows, orphan_pixels) float64 52.41 52.41 ... nan nan
    longitude_in         (rows, columns) float64 -88.58 -88.56 ... -72.82 -72.81
    longitude_orphan_in  (rows, orphan_pixels) float64 -87.42 -87.39 ... nan nan
Attributes: (12/16)
    absolute_orbit_number:  17380
    comment:                 
    contact:                eosupport@copernicus.esa.int
    creation_time:          20210828T012330Z
    history:                  2021-08-28T01:23:30Z: PUGCoreProcessor /data/ip...
    institution:            LN2
    ...                     ...
    source:                 IPF-SL-2 06.18
    start_offset:           54882
    start_time:             2021-08-26T15:40:26.116731Z
    stop_time:              2021-08-26T15:43:25.810293Z
    title:                  S3 SLSTR L1 Radiance and Brightness Temperatures ...
    track_offset:           998

In [78]:
import pandas

data = (
    pandas.DataFrame(
        {
            "longitude": geo.longitude_in.data.ravel(),
            "latitude": geo.latitude_in.data.ravel(),
            "lst": dataset.LST.load().data.ravel(),
        }
    )
    .dropna()
)
data

,longitude,latitude,lst
50,-87.829622,52.441963,286.462006
51,-87.829622,52.441963,286.462006
52,-87.809334,52.440007,285.954010
53,-87.799317,52.440464,285.936005
54,-87.779427,52.438736,285.779999
...,...,...,...
1799851,-74.454335,39.517924,303.563995
1799852,-74.441046,39.510563,304.290009
1799853,-74.426965,39.511543,302.660004
1799854,-74.412819,39.512522,306.609985


In [73]:
# import shapely.geometry
# from cartopy.crs import PlateCarree
# from cartopy.feature import BORDERS
# import matplotlib.pyplot as plt

# fig, ax = plt.subplots(figsize=(12, 12), subplot_kw=dict(projection=PlateCarree()))

# ax.add_geometries(
#     shapely.geometry.shape(item.geometry), crs=PlateCarree(), color="coral", alpha=0.1
# )

# data.plot.scatter(
#     x="longitude",
#     y="latitude",
#     c="lst",
#     ax=ax,
#     colormap="viridis",
#     marker=".",
#     alpha=0.2,
# )

# lower_left = (40.75, -74.01)
# upper_right = (40.88, -73.86)


# # ax.set(ybound=(20, 50), xbound=(-80, -70))
# ax.coastlines()
# ax.add_feature(BORDERS, linestyle="-")
# ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False);

In [74]:
submission_file = pandas.read_csv("../data/Submission_template.csv")
submission_file

,Longitude,Latitude,UHI Index
0,-73.971665,40.788763,NaN
1,-73.971928,40.788875,NaN
2,-73.967080,40.789080,NaN
3,-73.972550,40.789082,NaN
4,-73.969697,40.787953,NaN
...,...,...,...
1035,-73.919388,40.813803,NaN
1036,-73.931033,40.833178,NaN
1037,-73.934647,40.854542,NaN
1038,-73.917223,40.815413,NaN


In [75]:
merged_df = submission_file.merge(data, left_on=['Longitude', 'Latitude'], right_on=['longitude', 'latitude'], how='left')

merged_df.head()


,Longitude,Latitude,UHI Index,longitude,latitude,lst
0,-73.971665,40.788763,NaN,NaN,NaN,NaN
1,-73.971928,40.788875,NaN,NaN,NaN,NaN
2,-73.967080,40.789080,NaN,NaN,NaN,NaN
3,-73.972550,40.789082,NaN,NaN,NaN,NaN
4,-73.969697,40.787953,NaN,NaN,NaN,NaN
